## Sinhala Sentiment Analysis
This notebook covers:
- Loading Sinhala sentiment dataset
- Data preprocessing
- Text cleaning
- Label encoding
- Tokenization
- Padding sequences
- Preparing data for deep learning

## Import Required Libraries

In [4]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

import matplotlib.pyplot as plt

### Load Dataset

In [6]:
train_df = pd.read_csv(
    "../data/raw/train.tsv",
    sep="\t")

test_df = pd.read_csv(
    "../data/raw/test.tsv",
    sep="\t")

### Show train dataset size

In [7]:
train_df.shape

(7229, 10)

### Show test dataset size

In [8]:
test_df.shape

(1819, 10)

### Preview Train Dataset

In [9]:
train_df.head()

,id,article_id,title,author,date,body,comment_author,comment_date,comment_phrase,comment_sentiment
0,sa_train_0001,103990,සරදියල් ඇල්ලූ පී.සී. සභාත් සමරයි,රොමේෂ් ධනුෂ්ක සිල්වා,2013 මාර්තු මස 21 10:57:01,149 වැනි පොලිස් විරු සමරු දිනයේ එහි ප‍්‍රධාන ස...,ලලිත්,2013-03-21 11:03:41,සරදියෙල්වත් සමරුවොත් නරකද? ඔහුත් දුප්පතාට සේවය...,NEUTRAL
1,sa_train_0002,103990,සරදියල් ඇල්ලූ පී.සී. සභාත් සමරයි,රොමේෂ් ධනුෂ්ක සිල්වා,2013 මාර්තු මස 21 10:57:01,149 වැනි පොලිස් විරු සමරු දිනයේ එහි ප‍්‍රධාන ස...,නිවන්ත,2013-03-21 11:30:20,සරදියෙල්ටත් උපහාර කළා නම් තමා හොද,NEUTRAL
2,sa_train_0003,103990,සරදියල් ඇල්ලූ පී.සී. සභාත් සමරයි,රොමේෂ් ධනුෂ්ක සිල්වා,2013 මාර්තු මස 21 10:57:01,149 වැනි පොලිස් විරු සමරු දිනයේ එහි ප‍්‍රධාන ස...,අමා,2013-03-21 11:49:06,හොද වැඩක්,POSITIVE
3,sa_train_0004,103990,සරදියල් ඇල්ලූ පී.සී. සභාත් සමරයි,රොමේෂ් ධනුෂ්ක සිල්වා,2013 මාර්තු මස 21 10:57:01,149 වැනි පොලිස් විරු සමරු දිනයේ එහි ප‍්‍රධාන ස...,අනුර,2013-03-21 12:00:34,අපේ නමස්කාරය ඔබට!,POSITIVE
4,sa_train_0005,103990,සරදියල් ඇල්ලූ පී.සී. සභාත් සමරයි,රොමේෂ් ධනුෂ්ක සිල්වා,2013 මාර්තු මස 21 10:57:01,149 වැනි පොලිස් විරු සමරු දිනයේ එහි ප‍්‍රධාන ස...,මහින්ද,2013-03-21 12:27:24,මහත්තයා නරුනත් නාමය අමරණීයයි,POSITIVE


### Preview Test Dataset

In [10]:
test_df.head()

,id,article_id,title,author,date,body,comment_author,comment_date,comment_phrase,comment_sentiment
0,sa_test_0001,105812,රුවන්වැලි මහා සෑය හුණු පිරියම් කරන හැටි,ලංකාදීප කර්තෘ මණ්ඩලය,2013 මාර්තු මස 31 13:44:03,\n \nකුඩා දිය බුබුලක් අහස සිඹින මහා සෑයක් වී ...,පාලිත,2013-04-01 12:17:30,මෙවන් දේ කියවන්න දකින්න ලැබීම මහ පිනක්.,POSITIVE
1,sa_test_0002,105812,රුවන්වැලි මහා සෑය හුණු පිරියම් කරන හැටි,ලංකාදීප කර්තෘ මණ්ඩලය,2013 මාර්තු මස 31 13:44:03,\n \nකුඩා දිය බුබුලක් අහස සිඹින මහා සෑයක් වී ...,ප්‍රදීපා,2013-04-01 12:17:49,මහා සත්කාරයකට දායකවෙන ඔබ සැමට ජය ශ්‍රී මහා බෝ ...,POSITIVE
2,sa_test_0003,105812,රුවන්වැලි මහා සෑය හුණු පිරියම් කරන හැටි,ලංකාදීප කර්තෘ මණ්ඩලය,2013 මාර්තු මස 31 13:44:03,\n \nකුඩා දිය බුබුලක් අහස සිඹින මහා සෑයක් වී ...,ජයී තිසර,2013-04-01 13:58:24,තම ජීවිත නොතකුවේය සිදුවිය හැකි අනතුරේ.. දිනපතා...,POSITIVE
3,sa_test_0004,105812,රුවන්වැලි මහා සෑය හුණු පිරියම් කරන හැටි,ලංකාදීප කර්තෘ මණ්ඩලය,2013 මාර්තු මස 31 13:44:03,\n \nකුඩා දිය බුබුලක් අහස සිඹින මහා සෑයක් වී ...,කසුන් සදරුවන්,2013-04-01 14:37:50,අපිටත් දායකවෙන්න පුලුවන් නම් ... සාදු සාදු සාදු !,POSITIVE
4,sa_test_0005,105812,රුවන්වැලි මහා සෑය හුණු පිරියම් කරන හැටි,ලංකාදීප කර්තෘ මණ්ඩලය,2013 මාර්තු මස 31 13:44:03,\n \nකුඩා දිය බුබුලක් අහස සිඹින මහා සෑයක් වී ...,දිසානායක,2013-04-01 15:01:28,විශ්වයේ තිබෙන බුද්ධ රෂ්මි බලයෙන්ද සම්‍යක් දෘෂ්...,POSITIVE


### Check Dataset Information

In [11]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7229 entries, 0 to 7228
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 7229 non-null   str  
 1   article_id         7229 non-null   int64
 2   title              7229 non-null   str  
 3   author             7026 non-null   str  
 4   date               7229 non-null   str  
 5   body               7229 non-null   str  
 6   comment_author     7227 non-null   str  
 7   comment_date       7229 non-null   str  
 8   comment_phrase     7229 non-null   str  
 9   comment_sentiment  7229 non-null   str  
dtypes: int64(1), str(9)
memory usage: 34.6 MB


In [12]:
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1819 entries, 0 to 1818
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 1819 non-null   str  
 1   article_id         1819 non-null   int64
 2   title              1819 non-null   str  
 3   author             1724 non-null   str  
 4   date               1819 non-null   str  
 5   body               1788 non-null   str  
 6   comment_author     1819 non-null   str  
 7   comment_date       1819 non-null   str  
 8   comment_phrase     1819 non-null   str  
 9   comment_sentiment  1819 non-null   str  
dtypes: int64(1), str(9)
memory usage: 8.3 MB


### Check Sentiment Distribution

In [15]:
train_df["comment_sentiment"].value_counts()

comment_sentiment
NEUTRAL     3297
NEGATIVE    2067
POSITIVE    1865
Name: count, dtype: int64

### Remove Conflict Samples

In [16]:
train_df = train_df[
    train_df["comment_sentiment"] != "conflict"
]

test_df = test_df[
    test_df["comment_sentiment"] != "conflict"
]

print("Updated Train Shape:", train_df.shape)
print("Updated Test Shape:", test_df.shape)

Updated Train Shape: (7229, 10)
Updated Test Shape: (1819, 10)


### Updated Sentiment Distribution

In [17]:
train_df["comment_sentiment"].value_counts()

comment_sentiment
NEUTRAL     3297
NEGATIVE    2067
POSITIVE    1865
Name: count, dtype: int64

## ✂️ Select Required Columns

Keep only:
- comment_phrase
- comment_sentiment

Remove unnecessary metadata columns.

In [18]:
train_df = train_df[
    ["comment_phrase", "comment_sentiment"]
]

test_df = test_df[
    ["comment_phrase", "comment_sentiment"]
]

### Show Updated Dataset

In [19]:
train_df.head()

,comment_phrase,comment_sentiment
0,සරදියෙල්වත් සමරුවොත් නරකද? ඔහුත් දුප්පතාට සේවය...,NEUTRAL
1,සරදියෙල්ටත් උපහාර කළා නම් තමා හොද,NEUTRAL
2,හොද වැඩක්,POSITIVE
3,අපේ නමස්කාරය ඔබට!,POSITIVE
4,මහත්තයා නරුනත් නාමය අමරණීයයි,POSITIVE


In [20]:
test_df.head()

,comment_phrase,comment_sentiment
0,මෙවන් දේ කියවන්න දකින්න ලැබීම මහ පිනක්.,POSITIVE
1,මහා සත්කාරයකට දායකවෙන ඔබ සැමට ජය ශ්‍රී මහා බෝ ...,POSITIVE
2,තම ජීවිත නොතකුවේය සිදුවිය හැකි අනතුරේ.. දිනපතා...,POSITIVE
3,අපිටත් දායකවෙන්න පුලුවන් නම් ... සාදු සාදු සාදු !,POSITIVE
4,විශ්වයේ තිබෙන බුද්ධ රෂ්මි බලයෙන්ද සම්‍යක් දෘෂ්...,POSITIVE


In [21]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7229 entries, 0 to 7228
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   comment_phrase     7229 non-null   str  
 1   comment_sentiment  7229 non-null   str  
dtypes: str(2)
memory usage: 2.0 MB


In [22]:
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1819 entries, 0 to 1818
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   comment_phrase     1819 non-null   str  
 1   comment_sentiment  1819 non-null   str  
dtypes: str(2)
memory usage: 521.4 KB


### Check Missing Values

In [24]:
print(train_df.isnull().sum())

print("\n----------------------------\n")

print(test_df.isnull().sum())

comment_phrase       0
comment_sentiment    0
dtype: int64

----------------------------

comment_phrase       0
comment_sentiment    0
dtype: int64


## # 🧼 Sinhala Text Cleaning

Clean text by:
- removing special characters
- removing numbers
- removing extra spaces
- converting text to lowercase

### Create Text Cleaning Function

In [59]:
import re

def clean_text(text):

    text = str(text)

    # Keep Sinhala Unicode + English letters + spaces
    text = re.sub(r'[^a-zA-Z\u0D80-\u0DFF\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

### Clean Dataset Text

In [60]:
train_df["clean_text"] = train_df["comment_phrase"].apply(clean_text)

test_df["clean_text"] = test_df["comment_phrase"].apply(clean_text)

### Show Cleaned Text

In [61]:
train_df[
    ["comment_phrase", "clean_text"]
].head()

,comment_phrase,clean_text
0,සරදියෙල්වත් සමරුවොත් නරකද? ඔහුත් දුප්පතාට සේවය...,සරදියෙල්වත් සමරුවොත් නරකද ඔහුත් දුප්පතාට සේවය ...
1,සරදියෙල්ටත් උපහාර කළා නම් තමා හොද,සරදියෙල්ටත් උපහාර කළා නම් තමා හොද
2,හොද වැඩක්,හොද වැඩක්
3,අපේ නමස්කාරය ඔබට!,අපේ නමස්කාරය ඔබට
4,මහත්තයා නරුනත් නාමය අමරණීයයි,මහත්තයා නරුනත් නාමය අමරණීයයි


### Encode Labels

In [62]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

train_df["encoded_label"] = label_encoder.fit_transform(
    train_df["comment_sentiment"]
)

test_df["encoded_label"] = label_encoder.transform(
    test_df["comment_sentiment"]
)

### Display Label Mapping

In [63]:
label_mapping = dict(
    zip(
        label_encoder.classes_,
        label_encoder.transform(label_encoder.classes_)
    )
)

print(label_mapping)

{'NEGATIVE': np.int64(0), 'NEUTRAL': np.int64(1), 'POSITIVE': np.int64(2)}


## 🔤 Tokenization

Convert Sinhala text into integer sequences.

In [71]:
from tensorflow.keras.preprocessing.text import Tokenizer

MAX_WORDS = 8000

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(
    train_df["clean_text"]
)

### 🔢 Convert Text into Sequences

In [72]:
X_train = tokenizer.texts_to_sequences(
    train_df["clean_text"]
)

X_test = tokenizer.texts_to_sequences(
    test_df["clean_text"]
)

### 📏 Padding Sequences

Ensure all sequences have the same length.

In [73]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LENGTH = 80

X_train = pad_sequences(
    X_train,
    maxlen=MAX_LENGTH,
    padding="post"
)

X_test = pad_sequences(
    X_test,
    maxlen=MAX_LENGTH,
    padding="post"
)

### 🎯 Prepare Final Labels

In [74]:
y_train = train_df["encoded_label"]

y_test = test_df["encoded_label"]

print("Training Labels Shape:", y_train.shape)
print("Testing Labels Shape:", y_test.shape)

Training Labels Shape: (7229,)
Testing Labels Shape: (1819,)


### 📦 Final Dataset Shapes

In [75]:
print("X_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)

print("y_train Shape:", y_train.shape)
print("y_test Shape:", y_test.shape)

X_train Shape: (7229, 80)
X_test Shape: (1819, 80)
y_train Shape: (7229,)
y_test Shape: (1819,)


### Save Preprocessed Dataset

In [76]:
train_df.to_csv(
    "../data/preprocessed/train_preprocessed.csv",
    index=False
)

test_df.to_csv(
    "../data/preprocessed/test_preprocessed.csv",
    index=False
)

### Save Tokenizer and Label Encoder

In [77]:
import pickle

# Save tokenizer
with open(
    "../models/tokenizer.pkl",
    "wb"
) as file:
    pickle.dump(tokenizer, file)

# Save label encoder
with open(
    "../models/label_encoder.pkl",
    "wb"
) as file:
    pickle.dump(label_encoder, file)

### ✅ Data Preprocessing Completed

The dataset is now ready for:
- Embedding layer
- BiLSTM model
- Deep learning training
- Sentiment classification